ilk basda python ve sqli ise salaq

In [13]:
import sys
from pathlib import Path
import pandas as pd

from sqlalchemy import text, Unicode

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src" / "db.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.db import get_engine

engine = get_engine()

print("Python və SQL Server hazırdır!")

Python və SQL Server hazırdır!


csv fayllarimizi python data frame formasina getirek

In [14]:
DATA_DIR = PROJECT_ROOT / "data" / "bank_data"

customers = pd.read_csv(DATA_DIR / "customers.csv")
loans = pd.read_csv(DATA_DIR / "loans.csv")
transactions = pd.read_csv(DATA_DIR / "transactions.csv")

print("Customers:", customers.shape)
print("Loans:", loans.shape)
print("Transactions:", transactions.shape)

Customers: (5000, 13)
Loans: (4820, 10)
Transactions: (77396, 8)


fayllari sql servere yazaq

In [15]:
customers.to_sql("customers", engine, schema="bronze", if_exists="replace",
    index=False, dtype={c: Unicode(255) for c in customers.select_dtypes(include=["object", "string"]).columns})

loans.to_sql("loans", engine, schema="bronze", if_exists="replace",
    index=False, dtype={c: Unicode(255) for c in loans.select_dtypes(include=["object", "string"]).columns})

transactions.to_sql("transactions", engine, schema="bronze", if_exists="replace",
    index=False, dtype={c: Unicode(255) for c in transactions.select_dtypes(include=["object", "string"]).columns})

106

Kredit və müştəri məlumatlarını birləşdirək

In [16]:
with engine.connect() as conn:
    customers_df = pd.read_sql(
        text("SELECT * FROM bronze.customers"), conn
    )

    loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans"), conn
    )

bronze_loans_merged = customers_df.merge(
    loans_df,
    on="customer_id",
    how="left",
    validate="one_to_many"
)

bronze_loans_merged.head()

,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,...,churned,loan_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted
0,C00001,39,Qadın,Bakı,Yasamal,2,654,3156.63,908.98,2,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,0,L002507,4/3/2026,İstehlak krediti,1707.70,17.0,13.88,Rədd edildi,Natamam sənədlər,NaN
2,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,0,L003574,5/12/2026,İpoteka,137589.35,202.0,6.67,Rədd edildi,Yüksək borc yükü,NaN
3,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,0,L002552,4/6/2026,İstehlak krediti,14167.69,22.0,14.47,Təsdiqləndi,NaN,0.0
4,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,0,L003844,5/22/2026,İstehlak krediti,1067.37,20.0,14.33,Təsdiqləndi,NaN,0.0


Bayram təqvimini kreditlərə birləşdirək

In [17]:
# Bayram təqvimini Bronze-dan oxuyuruq
with engine.connect() as conn:
    calendar_df = pd.read_sql(
        text("SELECT * FROM bronze.calendar_reference"),
        conn
    )

# Tarixləri birləşdirmə üçün eyni formata gətiririk
bronze_loans_merged["merge_date"] = pd.to_datetime(
    bronze_loans_merged["application_date"]
).dt.date

calendar_df["merge_date"] = pd.to_datetime(
    calendar_df["date"]
).dt.date

# Bayram məlumatlarını kreditlərə əlavə edirik
bronze_loans_merged = bronze_loans_merged.merge(
    calendar_df[
        ["merge_date", "is_holiday", "holiday_name",
         "is_weekend", "is_workday", "is_pre_holiday"]
    ],
    on="merge_date",
    how="left",
    validate="many_to_one"
)
bronze_loans_merged.head()


,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,...,interest_rate,status,rejection_reason,defaulted,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,C00001,39,Qadın,Bakı,Yasamal,2,654,3156.63,908.98,2,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
1,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,13.88,Rədd edildi,Natamam sənədlər,NaN,2026-04-03,0.0,NaN,0.0,1.0,0.0
2,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,6.67,Rədd edildi,Yüksək borc yükü,NaN,2026-05-12,0.0,NaN,0.0,1.0,0.0
3,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,14.47,Təsdiqləndi,NaN,0.0,2026-04-06,0.0,NaN,0.0,1.0,0.0
4,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,14.33,Təsdiqləndi,NaN,0.0,2026-05-22,0.0,NaN,0.0,1.0,0.0


In [18]:
bronze_loans_merged.columns

Index(['customer_id', 'age', 'gender', 'city', 'branch', 'tenure_years',
       'credit_score', 'balance', 'salary', 'products_count', 'has_card',
       'is_active', 'churned', 'loan_id', 'application_date', 'product_type',
       'amount', 'term_months', 'interest_rate', 'status', 'rejection_reason',
       'defaulted', 'merge_date', 'is_holiday', 'holiday_name', 'is_weekend',
       'is_workday', 'is_pre_holiday'],
      dtype='str')

In [19]:
bronze_loans_merged.head()

,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,...,interest_rate,status,rejection_reason,defaulted,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,C00001,39,Qadın,Bakı,Yasamal,2,654,3156.63,908.98,2,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
1,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,13.88,Rədd edildi,Natamam sənədlər,NaN,2026-04-03,0.0,NaN,0.0,1.0,0.0
2,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,6.67,Rədd edildi,Yüksək borc yükü,NaN,2026-05-12,0.0,NaN,0.0,1.0,0.0
3,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,14.47,Təsdiqləndi,NaN,0.0,2026-04-06,0.0,NaN,0.0,1.0,0.0
4,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,14.33,Təsdiqləndi,NaN,0.0,2026-05-22,0.0,NaN,0.0,1.0,0.0


Hava məlumatlarını birləşdirək

In [20]:
weather_df = pd.read_sql("SELECT * FROM bronze.weather", engine)
weather_df["merge_date"] = pd.to_datetime(weather_df["time"]).dt.date

In [21]:
bronze_loans_merged = bronze_loans_merged.merge(
    weather_df, on=["merge_date", "city"], how="left"
)
bronze_loans_merged.head()

,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,...,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,C00001,39,Qadın,Bakı,Yasamal,2,654,3156.63,908.98,2,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,2026-04-03,0.0,NaN,0.0,1.0,0.0,2026-04-03,14.9,0.0,30.4
2,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,2026-05-12,0.0,NaN,0.0,1.0,0.0,2026-05-12,21.3,0.0,28.7
3,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,2026-04-06,0.0,NaN,0.0,1.0,0.0,2026-04-06,14.3,0.0,21.2
4,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,2026-05-22,0.0,NaN,0.0,1.0,0.0,2026-05-22,16.7,5.8,9.6


In [22]:
bronze_loans_merged.shape

(6891, 32)

Birləşdirilmiş cədvəli Bronze-a yazaq

In [23]:
# Birləşdirilmiş kredit məlumatlarını Bronze-a yazırıq
bronze_loans_merged.to_sql(
    name="loans_enriched",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False, 
    dtype={c: Unicode(255) for c in bronze_loans_merged.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

# SQL Server-də sətir sayını yoxlayırıq
with engine.connect() as conn:
    result = pd.read_sql(
        text("""
            SELECT COUNT(*) AS row_count
            FROM bronze.loans_enriched
        """),
        conn
    )

display(result)

,row_count
0,6891


Tranzaksiya məlumatlarını yoxlayaq

In [24]:
with engine.connect() as conn:
    transactions_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions"),
        conn
    )
transactions_df.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM


Transactions + Customers

In [25]:
# Tranzaksiyaları müştəri məlumatları ilə birləşdiririk
bronze_transactions_merged = transactions_df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

bronze_transactions_merged.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,has_card,is_active,churned
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,Bakı,Yasamal,11,653,3927.47,1568.44,1,0,1,1
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,Gəncə,Gəncə Mərkəz,7,647,3409.53,1133.34,1,0,1,1
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,Bakı,Binəqədi,5,783,0.00,973.82,2,1,1,0
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,Bakı,Yasamal,4,696,5133.15,1278.68,2,1,1,0
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,Lənkəran,Lənkəran Mərkəz,6,705,1443.03,2168.40,1,1,1,0


Transactions+ Calendar
Tranzaksiya ile Təqvim məlumatlarını birləşdirək
İndi tranzaksiyalara bayram, həftəsonu və iş günü məlumatlarını əlavə edəcəyik.

In [26]:
# Tranzaksiya tarixini birləşdirməyə hazırlayırıq
bronze_transactions_merged["merge_date"] = pd.to_datetime(
    bronze_transactions_merged["date"]
).dt.date

# Təqvim məlumatlarını birləşdiririk
bronze_transactions_merged = bronze_transactions_merged.merge(
    calendar_df[
        [
            "merge_date",
            "is_holiday",
            "holiday_name",
            "is_weekend",
            "is_workday",
            "is_pre_holiday"
        ]
    ],
    on="merge_date",
    how="left",
    validate="many_to_one"
)
bronze_transactions_merged.head()


,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,products_count,has_card,is_active,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,1,0,1,1,2026-01-01,1,Yeni il,0,0,0
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,1,0,1,1,2026-01-01,1,Yeni il,0,0,0
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,2,1,1,0,2026-01-01,1,Yeni il,0,0,0
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,2,1,1,0,2026-01-01,1,Yeni il,0,0,0
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,1,1,1,0,2026-01-01,1,Yeni il,0,0,0


Valyuta cədvəlini yoxlayaq
Bizdə bronze.exchange_rates cədvəlində AZN-dən USD və EUR-a məzənnələr var. Tranzaksiyalara bunları tarix və valyuta üzrə qoşacağıq.Amma əvvəlcə sütunları və valyutaları yoxlayaq.

In [27]:
with engine.connect() as conn:
    exchange_df = pd.read_sql(
        text("SELECT * FROM bronze.exchange_rates"),
        conn
    )
exchange_df.head()

,date,base,quote,rate
0,2026-01-01,AZN,EUR,0.50084
1,2026-01-01,AZN,USD,0.58837
2,2026-01-02,AZN,EUR,0.50095
3,2026-01-02,AZN,USD,0.58830
4,2026-01-03,AZN,EUR,0.50107


İndi məzənnələri tranzaksiyalara tarix və valyuta üzrə birləşdirəcəyik.

In [28]:
# Məzənnə tarixini uyğunlaşdırırıq
exchange_df["merge_date"] = pd.to_datetime(
    exchange_df["date"]
).dt.date

# Birləşdirmə üçün ayrıca cədvəl hazırlayırıq
rates_for_merge = exchange_df[
    ["merge_date", "quote", "base", "rate"]
].copy()

rates_for_merge = rates_for_merge.rename(
    columns={
        "quote": "rate_currency",
        "base": "rate_base"
    }
)

# Tranzaksiyalarla məzənnələri birləşdiririk
bronze_transactions_merged = bronze_transactions_merged.merge(
    rates_for_merge,
    left_on=["merge_date", "currency"],
    right_on=["merge_date", "rate_currency"],
    how="left",
    validate="many_to_one"
)

bronze_transactions_merged.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,1,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,1,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,0,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,0,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,0,2026-01-01,1,Yeni il,0,0,0,NaN,NaN,NaN


Hava məlumatlarını tranzaksiyalara əlavə edək

In [29]:
# Birləşdirmədən əvvəl sətir sayı
before_count = len(bronze_transactions_merged)

# Hava məlumatlarını əlavə edirik
bronze_transactions_merged = bronze_transactions_merged.merge(
    weather_df[
        [
            "merge_date",
            "city",
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ]
    ],
    on=["merge_date", "city"],
    how="left",
    validate="many_to_one"
)

bronze_transactions_merged.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,3.5,0.0,14.8
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,7.2,0.3,10.8


Birləşdirilmiş tranzaksiyaları Bronze-a yazaq

In [30]:
# Birləşdirilmiş tranzaksiyaları SQL Server-ə yazırıq
bronze_transactions_merged.to_sql(
    name="transactions_enriched",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False,
dtype={c: Unicode(255) for c in bronze_transactions_merged.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)



46

In [31]:
with engine.connect() as conn:
    transactions_enriched_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions_enriched"),
        conn
    )
transactions_enriched_df.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0040264,C02529,3/27/2026,19:55:59,9.91,AZN,Alış-veriş,POS,34,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
1,T0040265,C04195,3/27/2026,19:56:53,794.63,AZN,Nağdlaşdırma,ATM,45,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
2,T0040266,C03812,3/27/2026,19:57:15,120.90,AZN,Köçürmə,Mobil tətbiq,40,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
3,T0040267,C00782,3/27/2026,19:57:17,13.81,AZN,Alış-veriş,POS,48,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.0,14.3,13.8
4,T0040268,C02866,3/27/2026,19:57:21,36.76,AZN,Alış-veriş,POS,41,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,13.1,14.3,11.0


In [32]:
with engine.connect() as conn:
    loans_enriched_df = pd.read_sql(
        text("SELECT * FROM bronze.loans_enriched"),
        conn
    )
loans_enriched_df.head()

,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,...,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,C00001,39,Qadın,Bakı,Yasamal,2,654,3156.63,908.98,2,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,2026-04-03,0.0,NaN,0.0,1.0,0.0,2026-04-03,14.9,0.0,30.4
2,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,2026-05-12,0.0,NaN,0.0,1.0,0.0,2026-05-12,21.3,0.0,28.7
3,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,2026-04-06,0.0,NaN,0.0,1.0,0.0,2026-04-06,14.3,0.0,21.2
4,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,2026-05-22,0.0,NaN,0.0,1.0,0.0,2026-05-22,16.7,5.8,9.6


Bronze cədvəllərini yoxlayaq

In [33]:
with engine.connect() as conn:
    bronze_tables = pd.read_sql(
        text("""
            SELECT TABLE_NAME
            FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'bronze'
            ORDER BY TABLE_NAME
        """),
        conn
    )

display(bronze_tables)

,TABLE_NAME
0,calendar_reference
1,customers
2,exchange_rates
3,loans
4,loans_enriched
5,transactions
6,transactions_enriched
7,weather
